# Check Dataset class 

In [1]:
import os
import pandas as pd
from tqdm import tqdm
if os.getcwd().endswith("notebooks"):
    os.chdir("../")

from src.supervised_TCN.dataset import MotionFeatureDataset

In [2]:
processed_path = "data/processed"
paths = [os.path.join(processed_path, f) for f in os.listdir(processed_path) if f.endswith("10fps_processed.pkl")]

df_dict = {}
cols_to_keep = ['frame', 'hand_label', 'cx_smooth', 'cy_smooth']

# load df with where first column in csv serves as index
df_vid_name_map = pd.read_csv("data/scores/vid_name_map.csv", index_col=0)

with tqdm(total=len(paths), desc="Loading processed data") as pbar:
    for path in paths:
        vid = os.path.basename(path).replace("_10fps_processed.pkl", "")
        vid = vid.replace("hand_tracking_", "")
        participant_id = df_vid_name_map.loc[vid]['Participant Number']
        if int(participant_id) == 8:
            continue
        df_dict[(vid, int(participant_id))] = pd.read_pickle(path)[cols_to_keep]
        pbar.update(1)

Loading processed data:  97%|█████████▋| 83/86 [00:11<00:00,  7.35it/s]


In [3]:
df_scores = pd.read_csv("data/scores/merged_scores.csv")[['Vid_Name', 'QRS_Overal']]
grs_scores = df_scores.set_index('Vid_Name')['QRS_Overal'].to_dict()
grs_scores

{'2024-01-15_13-18-23': 48.5,
 '2024-01-15_13-37-36': 45.0,
 '2024-01-15_14-03-23': 60.5,
 '2024-01-15_14-32-45': 39.25,
 '2024-01-15_15-05-31': 38.0,
 '2024-01-15_15-38-13': 47.0,
 '2024-01-15_15-58-44': 62.0,
 '2024-01-15_16-17-19': 64.5,
 '2024-01-15_16-31-26': 63.0,
 '2024-01-15_17-37-24': 37.25,
 '2024-01-15_17-57-25': 40.5,
 '2024-01-15_18-17-18': 42.0,
 '2024-01-16_14-30-29': 36.0,
 '2024-01-16_15-20-49': 47.25,
 '2024-01-16_15-41-03': 43.0,
 '2024-01-16_16-03-11': 35.0,
 '2024-01-16_16-34-19': 44.25,
 '2024-01-16_17-00-33': 41.5,
 '2024-01-17_16-04-01': 44.0,
 '2024-01-17_16-22-28': 47.0,
 '2024-01-17_16-48-38': 41.25,
 '2024-01-18_10-40-40': 58.66666666666666,
 '2024-01-18_10-52-25': 62.66666666666666,
 '2024-01-18_11-08-40': 60.0,
 '2024-01-18_13-11-31': 60.75,
 '2024-01-18_13-25-00': 55.25,
 '2024-01-18_13-45-41': 62.25,
 '2024-01-18_14-39-24': 47.0,
 '2024-01-18_14-55-56': 52.25,
 '2024-01-18_15-17-27': 47.5,
 '2024-01-18_16-55-29': 59.75,
 '2024-01-18_17-08-23': 66.0,
 '20

In [4]:
df_scores['QRS_Overal'].describe()

count    83.000000
mean     49.938755
std       9.364622
min      33.750000
25%      43.125000
50%      49.250000
75%      58.708333
max      67.750000
Name: QRS_Overal, dtype: float64

# Single Training Split

In [5]:
from src.supervised_TCN.tcn import SurgicalTCN
from src.supervised_TCN.train import train_tcn, run_losocv
from torch.utils.data import DataLoader, Subset
import numpy as np

In [6]:
validation_goups = [1, 11, 17, 18, 27, 28]
train_groups = [g for g in range(1, 30) if (g not in validation_goups or g==8)]

train_df_dict = {key: df_dict[key] for key in df_dict.keys() if key[1] in train_groups}

training_stats = MotionFeatureDataset.compute_scaling_stats(
    df_dict=train_df_dict, 
    hand="Right", 
    orig_fps=30.0,
    max_gap_seconds=0.11
)


In [18]:
dataset = MotionFeatureDataset(df_dict, grs_scores, hand="Right", window_size=50, step_size=15, orig_fps=30.0, max_gap_seconds=0.11, device="cpu", scaling_stats=training_stats)

Processing Videos: 100%|██████████| 83/83 [00:00<00:00, 124.81it/s]


In [8]:
groups = np.array([item[3] for item in dataset.index_map])

train_indices = [i for i, g in enumerate(groups) if g in train_groups]
val_indices = [i for i, g in enumerate(groups) if g in validation_goups]

train_subset = Subset(dataset, train_indices)
val_subset = Subset(dataset, val_indices)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

In [9]:
model = SurgicalTCN(num_inputs=6, num_channels=[32, 32, 32], kernel_size=3, dropout=0.3)

In [10]:
best_model, best_val_corr = train_tcn(model, train_loader, val_loader, device="cpu", num_epochs=10)

Epoch 01 | Train Loss: 1795.6396 | Val Loss: 1070.7520 | Val Corr: 0.076
Epoch 02 | Train Loss: 526.7753 | Val Loss: 75.8723 | Val Corr: 0.181
Epoch 03 | Train Loss: 107.4763 | Val Loss: 55.1879 | Val Corr: 0.220
Epoch 04 | Train Loss: 87.8439 | Val Loss: 53.1216 | Val Corr: 0.265
Epoch 05 | Train Loss: 84.9182 | Val Loss: 53.5921 | Val Corr: 0.275
Epoch 06 | Train Loss: 83.6763 | Val Loss: 56.3547 | Val Corr: 0.235
Epoch 07 | Train Loss: 82.3391 | Val Loss: 60.0471 | Val Corr: 0.250
Epoch 08 | Train Loss: 81.5225 | Val Loss: 56.0762 | Val Corr: 0.257
Epoch 09 | Train Loss: 80.9962 | Val Loss: 53.8398 | Val Corr: 0.293
Epoch 10 | Train Loss: 80.2386 | Val Loss: 59.1489 | Val Corr: 0.253

Training complete.
Best validation correlation: 0.293


In [11]:
# save the best model
import torch
model_path = "results/models/SurgicalTCN3x32.pth"
torch.save(best_model.state_dict(), model_path)
print(f"Model saved to {model_path}")

Model saved to results/models/SurgicalTCN3x32.pth


# Convert into TCN encoder 

### - No regression Head.  
### - Returns the pooled feature vector (size 32) which serve as embeddings for subsequent MIL stage.

In [12]:
from src.supervised_TCN.tcn import SurgicalTCNEncoder

In [13]:
# 1. Instantiate the Encoder with the SAME hyperparameters as your best TCN
encoder = SurgicalTCNEncoder(
    num_inputs=6, 
    num_channels=[32, 32, 32], # Assuming this was your best config
    kernel_size=3, 
    dropout=0.0 # Dropout is usually turned off (eval mode) for extraction
)

# 2. Load the trained weights
# This transfers the learned filters from your supervised run
encoder.load_supervised_weights(model_path, device="cpu")
encoder.to("cpu")
encoder.eval()

Weights loaded. 
Missing keys (expected): [] 
Unexpected keys: ['linear.weight', 'linear.bias']


/Users/finnweikert/Desktop/Master Project/project/src/supervised_TCN/tcn.py:254: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(supervised_model_path,

SurgicalTCNEncoder(
  (network): Sequential(
    (0): TemporalBlock(
      (conv1): Conv1d(6, 32, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu1): LeakyReLU(negative_slope=0.01)
      (dropout1): Dropout(p=0.0, inplace=False)
      (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu2): LeakyReLU(negative_slope=0.01)
      (dropout2): Dropout(p=0.0, inplace=False)
      (downsample): Conv1d(6, 32, kernel_size=(1,), stride=(1,))
      (bn_downsample): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): TemporalBlock(
      (conv1): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu1): LeakyReLU(ne

In [14]:
model_path = "results/models/TCN_encoder3x32.pth"
torch.save(encoder.state_dict(), model_path)
print(f"Model saved to {model_path}")

Model saved to results/models/TCN_encoder3x32.pth


# Prepare Data for MIL training

In [15]:
from src.utils.utils import mil_collate_fn
from src.utils.utils import generate_mil_bags_cpu

In [19]:
data_loader_train = DataLoader(
    train_subset, 
    batch_size=32, 
    shuffle=False,
    collate_fn=mil_collate_fn
)

data_loader_val = DataLoader(
    val_subset, 
    batch_size=32, 
    shuffle=False,
    collate_fn=mil_collate_fn
)

data_loader = DataLoader(
    dataset, 
    batch_size=32, 
    shuffle=False,
    collate_fn=mil_collate_fn
)


In [20]:
video_window_embeddings_train = generate_mil_bags_cpu(data_loader_train, encoder)
video_window_embeddings_val = generate_mil_bags_cpu(data_loader_val, encoder)
video_window_embeddings = generate_mil_bags_cpu(data_loader, encoder)
# save the extracted bags for later MIL training

torch.save(video_window_embeddings_train, "data/embeddings/video_tcn_embeddings_train.pt")
torch.save(video_window_embeddings_val, "data/embeddings/video_tcn_embeddings_val.pt")
torch.save(video_window_embeddings, "data/embeddings/video_tcn_embeddings.pt")